## Grupo 14. ETL de la información obtenida de Inside Airbnb
### Proyecto: Riesgo de hipervalorización inmobiliaria por gentrificación turística en Bogotá
#### Objetivo
Este notebook documenta una primera exploración de la información obtenida desde Inside Aribnb y su correspondiente proceso de ETL para guardar en DVC. La transformación aplicada es necesaria para poder agregar esta informacion al dataset principal cuya llave será Barrio y Localidad.

### Librerias

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt ## creo que deben instalarlo en sus venv
import seaborn as sns ## creo que deben instalarlo en sus venv

### Datos - matriz_barrios_unificada.csv

In [2]:
matriz="../data/listings_barrio_localidad.csv"
df= pd.read_csv(matriz)
display(df.head())

,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,...,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license,barrio,coincidencias_barrio,localidad,coincidencias_localidad,geometry
0,27841,Rosales Luxury Large 2BR Boutique Apt,113734,1462509905126026119,Marcela,NaN,Chapinero,4.657580,-74.053260,Entire home/apt,...,0.23,3,168,0,112727.0,BELLAVISTA,1,CHAPINERO,1,POINT (-74.05326 4.65758)
1,65762,Apartaestudios La Candelaria,321447,1462515641808590179,Teresa,NaN,Candelaria,4.594850,-74.071890,Entire home/apt,...,0.19,24,258,0,100613.0,EGIPTO,1,CANDELARIA,1,POINT (-74.07189 4.59485)
2,124029,Exclusive Loft in 93 park w Gym,619048,1462513052712343807,Jaime,NaN,Chapinero,4.679779,-74.055817,Entire home/apt,...,0.28,2,289,0,69770.0,CHICO NORTE,1,CHAPINERO,1,POINT (-74.055817 4.679779)
3,130427,Feel like home @ nice & new loft,642197,1462524657755403478,Jaime,NaN,Suba,4.695060,-74.061650,Entire home/apt,...,0.01,1,365,0,NaN,PASADENA,1,SUBA,1,POINT (-74.06165 4.69506)
4,138660,Excelent Studio ROSALES/ ZONA G,676823,1462513470872353935,Carlos,NaN,Chapinero,4.648440,-74.055270,Entire home/apt,...,0.14,1,225,0,NaN,GRANADA,1,CHAPINERO,1,POINT (-74.05527 4.64844)


In [3]:
print("Dimensiones de la matriz:", df.shape)
print("\nNombres de las columnas:")
print(df.columns.tolist())

Dimensiones de la matriz: (19505, 24)

Nombres de las columnas:
['id', 'name', 'host_id', 'host_profile_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'number_of_reviews_ltm', 'license', 'barrio', 'coincidencias_barrio', 'localidad', 'coincidencias_localidad', 'geometry']


### Revisión de la calidad de los datos
Revisamos los tipos de datos, los valores faltantes, las filas exactamente duplicadas y las estadísticas descriptivas. Esta revisión permite detectar problemas que podrían distorsionar las comparaciones o el entrenamiento posterior.

In [4]:
# Tipos de datos
print("Tipos de variables:")
display(df.dtypes.to_frame("tipo"))

# Valores faltantes
faltantes = pd.DataFrame({
    "cantidad": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)})

print("Valores faltantes:")
display(faltantes)

# Filas duplicadas
print("Filas duplicadas:", df.duplicated().sum())

# Estadísticas de variables numéricas
print("Estadísticas descriptivas:")
display(df.describe().T)

Tipos de variables:


,tipo
id,int64
name,object
host_id,int64
host_profile_id,int64
host_name,object
neighbourhood_group,float64
neighbourhood,object
latitude,float64
longitude,float64
room_type,object


Valores faltantes:


,cantidad,porcentaje
id,0,0.00
name,0,0.00
host_id,0,0.00
host_profile_id,0,0.00
host_name,0,0.00
neighbourhood_group,19505,100.00
neighbourhood,0,0.00
latitude,0,0.00
longitude,0,0.00
room_type,0,0.00


Filas duplicadas: 0
Estadísticas descriptivas:


,count,mean,std,min,25%,50%,75%,max
id,19505.0,9.922371e+17,6.179715e+17,2.784100e+04,6.078547e+17,1.161773e+18,1.516893e+18,1.774233e+18
host_id,19505.0,3.358639e+16,2.383178e+17,2.516700e+04,1.026241e+08,2.761532e+08,4.999131e+08,1.772701e+18
host_profile_id,19505.0,1.483024e+18,5.360774e+16,1.462507e+18,1.463883e+18,1.469179e+18,1.470287e+18,1.773670e+18
neighbourhood_group,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,19505.0,4.663147e+00,3.963360e-02,4.028166e+00,4.635930e+00,4.666795e+00,4.688510e+00,4.822180e+00
longitude,19505.0,-7.407047e+01,2.768949e-02,-7.431473e+01,-7.408022e+01,-7.406524e+01,-7.405296e+01,-7.400389e+01
price,18794.0,2.928784e+05,3.849321e+06,1.540200e+04,1.098162e+05,1.607720e+05,2.294878e+05,2.585147e+08
minimum_nights,19498.0,7.007232e+00,1.273582e+01,1.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00,3.650000e+02
number_of_reviews,19505.0,2.915534e+01,7.645465e+01,0.000000e+00,1.000000e+00,8.000000e+00,3.400000e+01,6.340000e+03
reviews_per_month,15416.0,1.489272e+00,2.135473e+00,1.000000e-02,2.800000e-01,8.650000e-01,2.000000e+00,7.226000e+01


### Transformación

In [6]:
df_agrupado = df.groupby(['localidad', 'barrio']).size().reset_index(name='conteo')

In [8]:
df_agrupado.head()

,localidad,barrio,conteo
0,ANTONIO NARIÑO,CIUDAD BERNA,1
1,ANTONIO NARIÑO,CIUDAD JARDIN SUR,4
2,ANTONIO NARIÑO,EDUARDO FREY,5
3,ANTONIO NARIÑO,LA FRAGUA,1
4,ANTONIO NARIÑO,LA FRAGUITA,7


In [9]:
df_agrupado.describe()

,conteo
count,654.000000
mean,29.821101
std,75.180498
min,1.000000
25%,2.000000
50%,7.000000
75%,21.000000
max,883.000000


In [13]:
df_agrupado.to_csv('../data/listings_agrupado_barrio_localidad.csv', index=False, encoding='utf-8')